# Lab Assignment 9 — 3D CNN for Automated Fall Detection (URFD)

**Course:** AI in Healthcare · **Task:** Binary classification of video clips into `Fall` and `Non-fall / ADL`.

This notebook is designed for **Google Colab** or a local Python environment. It downloads/loads the URFD dataset, checks video files, samples fixed-length RGB clips, creates a leakage-aware stratified split, trains a 3D CNN, and reports Accuracy, Precision, Recall, F1, AUROC, a confusion matrix, and an ROC curve.

> **Important:** URFD has only about 70 sequences, so a high score is not guaranteed and estimates will have substantial uncertainty. The notebook uses video-level splitting (not frame-level splitting) to reduce leakage. Treat results as an educational experiment, not a clinically validated fall-detection system.

## 0. Runtime and dataset setup

**Recommended:** Google Colab → Runtime → Change runtime type → **T4 GPU** (or another available GPU).

Dataset page: [UR Fall Detection Dataset on Kaggle](https://www.kaggle.com/datasets/shahliza27/ur-fall-detection-dataset)

### Option A — Kaggle API download
1. In Kaggle, open **Settings → API → Create New Token**. This downloads `kaggle.json`.
2. Run the cell below, upload `kaggle.json`, then download and unzip the dataset.
3. Keep your Kaggle token private; do not share the notebook with the token embedded.

### Option B — Upload/extract the dataset yourself
If the Kaggle download command does not work, download the dataset from the page in your browser, extract it, and set `DATA_ROOT` in the configuration cell to the extracted folder.

The notebook searches subfolders recursively and uses filenames/folder names containing `fall` or `adl`/`nonfall`/`non-fall` to infer labels. It prefers RGB/color videos when filenames indicate modality. Review the discovered manifest before training.

### Fix included for label detection
A common error is labeling every video as Fall because the dataset root itself contains the word “fall”. This version ignores generic dataset-root names and labels only from video filenames or explicit class folders. Check the class counts before training.


In [ ]:
# Run this cell in Colab for Kaggle download. If you already have the data, skip it.
from pathlib import Path
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # select kaggle.json
    if "kaggle.json" in uploaded:
        os.makedirs("/root/.kaggle", exist_ok=True)
        Path("/root/.kaggle/kaggle.json").write_bytes(uploaded["kaggle.json"])
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        os.makedirs("/content/urfd", exist_ok=True)
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", "shahliza27/ur-fall-detection-dataset",
             "-p", "/content/urfd", "--unzip"],
            text=True, capture_output=True
        )
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            print("If download failed, download the dataset manually from Kaggle and extract it into /content/urfd.")
    else:
        print("No kaggle.json uploaded. You can manually extract the dataset and set DATA_ROOT below.")
else:
    print("Local runtime detected. Download/extract the dataset manually, or use Kaggle CLI after configuring credentials.")

In [ ]:
# Install dependencies (safe to rerun)
%pip -q install opencv-python-headless scikit-learn matplotlib pandas tqdm

## 1. Imports, configuration, and hardware check

In [ ]:
import os, re, random, math, json, time, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

# Change this if your dataset is stored elsewhere.
DATA_ROOT = Path("/content/urfd") if Path("/content/urfd").exists() else Path("./urfd")
# Video preprocessing
NUM_FRAMES = 16
IMAGE_SIZE = 112
BATCH_SIZE = 4
NUM_WORKERS = 2 if os.name != "nt" else 0
print("DATA_ROOT:", DATA_ROOT.resolve())

## 2. Discover videos, infer labels, and validate files

The expected labels are `1 = Fall`, `0 = Non-fall / ADL`. This cell:
- searches recursively for common video extensions;
- excludes depth/accelerometer files when filenames clearly identify RGB/color streams;
- infers labels from names and parent folders;
- opens each candidate with OpenCV and removes unreadable/empty files.

If the printed manifest has missing or incorrect labels, fix the relevant filenames or add label rules before continuing. Do not train on an unreviewed manifest.

In [ ]:
VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv", ".mpeg", ".mpg", ".wmv"}

# Infer from the filename, or from an explicit class folder only. Do NOT use
# generic parent/root names such as "ur-fall-detection-dataset" as a label.
def infer_label(path: Path):
    stem = re.sub(r"[^a-z0-9]+", " ", path.stem.lower()).strip()
    tokens = stem.split()
    if tokens and (tokens[0] in {"fall", "falls", "falling"} or re.match(r"^fall(?:ing)?\d", path.stem.lower())):
        return 1
    if tokens and tokens[0] == "adl":
        return 0
    if any(x in stem for x in ["non fall", "nonfall", "normal", "daily activity"]):
        return 0
    # Folder labels must be explicit and short; ignore dataset root names.
    for parent in list(path.parents)[:3]:
        folder = re.sub(r"[^a-z0-9]+", " ", parent.name.lower()).strip()
        if folder in {"fall", "falls", "falling", "fall videos", "fall video", "falls videos"}:
            return 1
        if folder in {"adl", "non fall", "nonfall", "normal", "adl videos", "adl video", "non fall videos"}:
            return 0
    return None

def is_rgb_candidate(path: Path):
    s = path.stem.lower()
    return not any(k in s for k in ["depth", "accel", "skeleton", "infrared", "infra_red", "ir_"])

def video_is_readable(path: Path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        cap.release()
        return False, 0, 0, 0, 0.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    ok, frame = cap.read()
    cap.release()
    return bool(ok and frame is not None and n > 0 and w > 0 and h > 0), n, w, h, fps

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATA_ROOT}. Extract URFD and set DATA_ROOT to the extracted folder.")

candidates = [p for p in DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS and is_rgb_candidate(p)]
rows, bad_files, unlabeled = [], [], []
for p in tqdm(candidates, desc="Checking videos"):
    label = infer_label(p)
    if label is None:
        unlabeled.append(str(p))
        continue
    ok, n, w, h, fps = video_is_readable(p)
    if not ok:
        bad_files.append(str(p))
        continue
    rows.append({"path": str(p), "label": int(label), "class": "Fall" if label else "Non-fall",
                 "frames": n, "width": w, "height": h,
                 "duration_s": round(n / fps, 2) if fps > 0 else None})

manifest = pd.DataFrame(rows).drop_duplicates(subset=["path"]).reset_index(drop=True)
print(f"Candidate video files: {len(candidates)}")
print(f"Usable labeled videos: {len(manifest)}")
print(f"Unreadable/corrupt videos skipped: {len(bad_files)}")
print(f"Unlabeled candidates skipped: {len(unlabeled)}")
if bad_files: print("Examples of corrupt files:", bad_files[:5])
if unlabeled: print("Examples of unlabeled files (review if expected):", unlabeled[:10])
if len(manifest):
    display(manifest.head(15))
    print("\\nClass counts:")
    print(manifest["class"].value_counts())
else:
    print("No videos were assigned labels. Candidate paths:", [str(p) for p in candidates[:25]])
    print("If filenames do not start with fall-/adl-, set labels manually using the folder names.")


### Sanity check: class balance and video samples

Confirm both classes are represented and the class labels match the actual videos. The sample grid below displays representative frames from discovered videos.

In [ ]:
if len(manifest) < 10 or manifest["label"].nunique() < 2:
    raise ValueError("Too few labeled videos or only one class found. Check the candidate paths and class counts above. Ensure filenames start with fall-/adl- or videos are inside explicit Fall/ADL folders.")
display(manifest.groupby("class").agg(videos=("path","count"), median_frames=("frames","median"), median_width=("width","median"), median_height=("height","median")))

def read_preview_frame(path, fraction=0.5):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, min(total-1, int(total*fraction)))
    ok, frame = cap.read()
    cap.release()
    if not ok: return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

sample_df = pd.concat([
    manifest[manifest.label == 0].head(3),
    manifest[manifest.label == 1].head(3)
]).sample(frac=1, random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):
    frame = read_preview_frame(row.path)
    if frame is not None:
        ax.imshow(frame)
    ax.set_title(f"{row['class']} | {Path(row.path).name[:24]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Split at the video level (70% / 15% / 15%)

All frames/clips from a source video stay within the same split. This is critical: randomly splitting frames from one video across train and test would inflate performance. With a small dataset, the test set is noisy; report its size alongside the metrics.

In [ ]:
train_df, temp_df = train_test_split(
    manifest, test_size=0.30, random_state=SEED, stratify=manifest["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)
for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(df)} videos | {df['class'].value_counts().to_dict()}")
assert set(train_df.path).isdisjoint(set(val_df.path))
assert set(train_df.path).isdisjoint(set(test_df.path))
assert set(val_df.path).isdisjoint(set(test_df.path))

## 4. Video clip extraction and augmentation

Each video becomes a tensor of shape `T × H × W × C` (`16 × 112 × 112 × 3`), then PyTorch receives `C × T × H × W`. Frames are sampled across the full video; training uses temporal jitter and consistent spatial augmentation across a clip. Validation/test preprocessing is deterministic.

For small datasets, aggressive augmentation can hurt as well as help. If your runtime is slow, reduce `IMAGE_SIZE` to 96 or `NUM_FRAMES` to 12.

In [ ]:
def decode_video(path, num_frames=NUM_FRAMES, size=IMAGE_SIZE, training=False):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total <= 0:
        cap.release()
        raise ValueError(f"No frames reported by video: {path}")
    # Select evenly spaced indices. Training jitters the sample positions slightly.
    if training and total > num_frames:
        base = np.linspace(0, total - 1, num_frames)
        max_jitter = max(1, int(total / num_frames * 0.35))
        indices = np.clip(base + np.random.randint(-max_jitter, max_jitter + 1, size=num_frames), 0, total-1)
        indices = np.sort(indices.astype(int))
    else:
        indices = np.linspace(0, max(total-1, 0), num_frames).astype(int)

    frames = []
    last_good = None
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok or frame is None:
            if last_good is None:
                # Try reading sequentially from start as a fallback.
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                ok, frame = cap.read()
                if not ok or frame is None:
                    cap.release()
                    raise ValueError(f"Could not decode frames from {path}")
            else:
                frame = last_good.copy()
        last_good = frame
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w = frame.shape[:2]
        # Resize while preserving aspect ratio, then center crop.
        scale = size / min(h, w)
        nh, nw = int(round(h*scale)), int(round(w*scale))
        frame = cv2.resize(frame, (nw, nh), interpolation=cv2.INTER_AREA)
        y0, x0 = (nh-size)//2, (nw-size)//2
        frame = frame[y0:y0+size, x0:x0+size]
        frames.append(frame)
    cap.release()
    arr = np.stack(frames).astype(np.float32) / 255.0  # T,H,W,C in [0,1]

    if training:
        # Same transform applied to all frames to preserve temporal consistency.
        if random.random() < 0.5:
            arr = arr[:, :, ::-1, :]
        # Small spatial crop/scale jitter, then resize back.
        if random.random() < 0.35:
            crop_frac = random.uniform(0.88, 0.98)
            crop = int(size * crop_frac)
            y0 = random.randint(0, size-crop)
            x0 = random.randint(0, size-crop)
            aug = []
            for f in arr:
                crop_img = (f[y0:y0+crop, x0:x0+crop] * 255).astype(np.uint8)
                aug.append(cv2.resize(crop_img, (size, size), interpolation=cv2.INTER_LINEAR))
            arr = np.stack(aug).astype(np.float32) / 255.0
        # Tiny brightness/contrast variation, consistent across the clip.
        if random.random() < 0.3:
            contrast = random.uniform(0.9, 1.1)
            brightness = random.uniform(-0.04, 0.04)
            arr = np.clip(arr * contrast + brightness, 0, 1)

    return arr

class URFDDataset(Dataset):
    def __init__(self, frame_df, training=False):
        self.df = frame_df.reset_index(drop=True)
        self.training = training
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            clip = decode_video(row.path, training=self.training)
        except Exception as e:
            # Corrupt files should already have been filtered. Raise a clear error rather than silently changing labels.
            raise RuntimeError(f"Failed to decode {row.path}: {e}") from e
        # T,H,W,C -> C,T,H,W; contiguous prevents negative-stride issues after horizontal flip.
        clip = np.ascontiguousarray(clip.transpose(3, 0, 1, 2), dtype=np.float32)
        return torch.from_numpy(clip), torch.tensor(int(row.label), dtype=torch.long), row.path

train_ds = URFDDataset(train_df, training=True)
val_ds = URFDDataset(val_df, training=False)
test_ds = URFDDataset(test_df, training=False)

# Weighted sampler addresses class imbalance in training.
train_labels = train_df.label.to_numpy()
counts = np.bincount(train_labels, minlength=2)
class_weights = 1.0 / np.maximum(counts, 1)
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))

xb, yb, paths = next(iter(train_loader))
print("Batch tensor:", tuple(xb.shape), "| expected B,C,T,H,W")
print("Pixel range:", float(xb.min()), float(xb.max()), "| labels:", yb.tolist())

## 5. Baseline 3D CNN

The network uses 3D convolutions to learn spatial and temporal patterns, batch normalization, ReLU, max pooling, dropout, and global average pooling. This is a compact baseline suitable for the small dataset; it is not expected to outperform a carefully tuned pretrained video model in every setting.

In [ ]:
class Conv3DBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(1,2,2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=pool)
        )
    def forward(self, x): return self.block(x)

class Fall3DCNN(nn.Module):
    def __init__(self, dropout=0.35):
        super().__init__()
        self.features = nn.Sequential(
            Conv3DBlock(3, 16, pool=(1,2,2)),
            Conv3DBlock(16, 32, pool=(2,2,2)),
            Conv3DBlock(32, 64, pool=(2,2,2)),
            Conv3DBlock(64, 96, pool=(2,2,2)),
        )
        self.pool = nn.AdaptiveAvgPool3d((1,1,1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(96, 48),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout/2),
            nn.Linear(48, 2)
        )
    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

model = Fall3DCNN().to(DEVICE)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 6. Training with early stopping

The training loop uses AdamW, a learning-rate scheduler, weighted cross-entropy, validation loss, and early stopping. Best weights are saved to `best_3dcnn_urfd.pt`. Since the dataset is small, results may vary with the random split and seed.

In [ ]:
EPOCHS = 35
PATIENCE = 7
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4

# Weighted loss complements the balanced sampler.
counts = np.bincount(train_df.label.to_numpy(), minlength=2)
loss_weights = torch.tensor((counts.sum() / (2*np.maximum(counts,1))), dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=loss_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

def run_epoch(loader, training=False):
    model.train(training)
    total_loss, all_y, all_prob, all_pred = 0.0, [], [], []
    for clips, labels, _ in loader:
        clips, labels = clips.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(clips)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = (probs >= 0.5).long()
        all_y.extend(labels.detach().cpu().numpy().tolist())
        all_prob.extend(probs.detach().cpu().numpy().tolist())
        all_pred.extend(preds.detach().cpu().numpy().tolist())
    avg_loss = total_loss / max(len(loader.dataset), 1)
    return avg_loss, np.array(all_y), np.array(all_prob), np.array(all_pred)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val, best_epoch, wait = float("inf"), 0, 0
BEST_PATH = Path("best_3dcnn_urfd.pt")

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_y, tr_prob, tr_pred = run_epoch(train_loader, training=True)
    va_loss, va_y, va_prob, va_pred = run_epoch(val_loader, training=False)
    scheduler.step(va_loss)
    tr_acc = accuracy_score(tr_y, tr_pred)
    va_acc = accuracy_score(va_y, va_pred)
    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc); history["val_acc"].append(va_acc)
    print(f"Epoch {epoch:02d}/{EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}")
    if va_loss < best_val:
        best_val, best_epoch, wait = va_loss, epoch, 0
        torch.save(model.state_dict(), BEST_PATH)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
            break

model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
print("Loaded best checkpoint from epoch", best_epoch, "| validation loss:", round(best_val, 4))

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Validation")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True, alpha=.3)
axes[1].plot(history["train_acc"], label="Train")
axes[1].plot(history["val_acc"], label="Validation")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True, alpha=.3)
plt.tight_layout(); plt.show()

## 7. Final held-out test evaluation

The test set is evaluated once after model selection. AUROC uses predicted fall probabilities; if the test set accidentally contains only one class, AUROC is undefined (the stratified split should usually prevent this when counts allow). For healthcare fall detection, **recall/sensitivity for falls** is especially important, but false alarms (precision) also matter.

In [ ]:
test_loss, y_true, y_prob, y_pred = run_epoch(test_loader, training=False)
metrics = {
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision (Fall)": precision_score(y_true, y_pred, zero_division=0),
    "Recall / Sensitivity (Fall)": recall_score(y_true, y_pred, zero_division=0),
    "F1-score (Fall)": f1_score(y_true, y_pred, zero_division=0),
    "AUROC": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else float("nan"),
    "Test loss": test_loss,
}
print("Held-out test metrics")
for k, v in metrics.items(): print(f"{k:28s}: {v:.4f}" if np.isfinite(v) else f"{k:28s}: undefined")
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["Non-fall", "Fall"], zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0,1])
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("Confusion Matrix (test videos)")
ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
ax.set_xticks([0,1], ["Non-fall", "Fall"]); ax.set_yticks([0,1], ["Non-fall", "Fall"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center")
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

if len(np.unique(y_true)) == 2:
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    auc_value = roc_auc_score(y_true, y_prob)
    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"3D CNN (AUC = {auc_value:.3f})")
    plt.plot([0,1], [0,1], linestyle="--", label="Chance")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curve (test videos)"); plt.legend(); plt.grid(True, alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print("ROC curve not plotted because the test set contains only one class.")

## 8. Inspect individual errors

This table lists test videos where the predicted class differs from the true class, with the model's fall probability. Review these clips manually for camera viewpoint, occlusion, motion blur, lighting, or activities that resemble a fall. The notebook cannot automatically infer the cause of each error from labels alone.

In [ ]:
test_results = test_df.reset_index(drop=True).copy()
test_results["true_label"] = y_true
test_results["true_class"] = test_results["true_label"].map({0:"Non-fall", 1:"Fall"})
test_results["fall_probability"] = y_prob
test_results["pred_label"] = y_pred
test_results["pred_class"] = test_results["pred_label"].map({0:"Non-fall", 1:"Fall"})
test_results["correct"] = test_results["true_label"] == test_results["pred_label"]
display(test_results[["path","true_class","pred_class","fall_probability","correct"]])
errors = test_results[~test_results.correct]
print(f"Misclassified videos: {len(errors)} / {len(test_results)}")

## 9. Save model, metrics, and predictions

The checkpoint and CSV files are saved in the current working directory. In Colab, download them from the Files pane or use the optional download cell.

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "num_frames": NUM_FRAMES,
    "image_size": IMAGE_SIZE,
    "class_mapping": {"Non-fall": 0, "Fall": 1},
    "seed": SEED,
}, "urfd_3dcnn_checkpoint.pt")
pd.DataFrame([metrics]).to_csv("urfd_test_metrics.csv", index=False)
test_results.to_csv("urfd_test_predictions.csv", index=False)
manifest.to_csv("urfd_video_manifest.csv", index=False)
print("Saved: urfd_3dcnn_checkpoint.pt, urfd_test_metrics.csv, urfd_test_predictions.csv, urfd_video_manifest.csv")

## 10. Discussion / conclusion (use your actual outputs)

**Suggested report structure**
1. **Objective:** Build a 3D CNN to classify fall and non-fall activity from short RGB video clips.
2. **Dataset:** Report the actual number of usable videos after file validation and class counts. Mention that URFD includes RGB/depth and sensor streams; this notebook uses RGB videos only.
3. **Preprocessing:** 16 evenly sampled frames per video, resize/center crop to 112×112, RGB conversion, normalization to [0,1], and training-only horizontal flip, crop/scale, brightness variation, and temporal jitter.
4. **Split:** Stratified 70/15/15 split by video, preventing clips from the same source video appearing in different partitions.
5. **Model:** 3D convolution blocks with batch normalization, ReLU, pooling, dropout, and a two-class output.
6. **Training:** AdamW, weighted cross-entropy, weighted random sampling, learning-rate reduction, and early stopping.
7. **Results:** Copy the measured test Accuracy, Precision, Recall, F1-score, AUROC, confusion matrix, and ROC curve from the notebook.
8. **Error analysis:** Use the error table and inspect clips. Discuss possible effects of camera angle, partial occlusion, lighting, blur, and ADL movements that resemble falling.
9. **Limitations:** Small dataset, one controlled environment, split sensitivity, possible subject/camera correlations, and no external validation. A high score on this dataset does not establish real-world clinical safety.

### Ways to make the experiment stronger
- Repeat the experiment across 3–5 random seeds and report mean ± standard deviation.
- If subject IDs are available, use subject-independent splits.
- Compare RGB against depth or RGB+depth only after verifying correct modality pairing.
- Compare with a pretrained video model (e.g., pretrained R(2+1)D) if network access and compute are available.
- Tune the decision threshold on validation data if sensitivity is the priority; never tune on the test set.
- Do not claim “great accuracy” unless it is actually supported by held-out evaluation. With only around 70 videos, uncertainty is high.